# Ammonia Cracking for Hydrogen Production

This notebook demonstrates simulation of ammonia (NH₃) thermal cracking
(decomposition) for hydrogen production using NeqSim's Gibbs reactor.

## Reaction

$$2\text{NH}_3 \rightleftharpoons \text{N}_2 + 3\text{H}_2 \quad \Delta H = +92 \text{ kJ/mol}$$

The reaction is:
- **Endothermic** — requires heat input
- **Favored at high temperature** (typically 400–900°C)
- **Favored at low pressure** (Le Chatelier's principle — 4 moles → 2 moles)
- **Catalyzed** by Ni, Ru, or Fe-based catalysts

## Topics Covered
1. Chemical equilibrium calculation at various temperatures
2. Effect of pressure on conversion
3. Complete cracker process with heat integration
4. Hydrogen purification (N₂ removal)

In [ ]:
# Setup NeqSim
import subprocess, sys
try:
    from neqsim import jneqsim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim'])
    from neqsim import jneqsim

import jpype
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Separator = jneqsim.process.equipment.separator.Separator
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
GibbsReactor = jneqsim.process.equipment.reactor.GibbsReactor
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

print('NeqSim loaded successfully')

## 1. Equilibrium Conversion vs Temperature

At equilibrium, the ammonia decomposition is governed by Gibbs free energy
minimization. We sweep temperature from 200°C to 900°C at constant pressure.

In [ ]:
# Temperature sweep for NH3 cracking equilibrium
temperatures_C = np.arange(200, 950, 25)
pressure = 10.0  # bara

conversions = []
h2_fractions = []
n2_fractions = []
nh3_fractions = []

for t in temperatures_C:
    try:
        # Create feed: pure ammonia
        feed_fluid = SystemSrkEos(273.15 + float(t), pressure)
        feed_fluid.addComponent('ammonia', 1.0)
        feed_fluid.addComponent('nitrogen', 1e-10)  # trace to enable
        feed_fluid.addComponent('hydrogen', 1e-10)  # trace to enable
        feed_fluid.setMixingRule('classic')

        feed = Stream('NH3 Feed', feed_fluid)
        feed.setFlowRate(1000.0, 'kg/hr')
        feed.setTemperature(float(t), 'C')
        feed.setPressure(pressure, 'bara')

        proc = ProcessSystem()
        proc.add(feed)

        reactor = GibbsReactor('Cracker', feed)
        reactor.setEnergyMode('isothermal')
        reactor.setMaxIterations(5000)
        reactor.setConvergenceTolerance(1e-6)
        proc.add(reactor)

        proc.run()

        # Get outlet composition
        outlet = reactor.getOutletStream()
        outlet_fluid = outlet.getFluid()

        # Read mole fractions of components
        x_nh3 = float(outlet_fluid.getPhase(0).getComponent('ammonia').getx())
        x_n2 = float(outlet_fluid.getPhase(0).getComponent('nitrogen').getx())
        x_h2 = float(outlet_fluid.getPhase(0).getComponent('hydrogen').getx())

        # Conversion = 1 - (NH3_out / NH3_in)
        # Since we started with ~100% NH3, conversion ≈ 1 - x_nh3
        conversion = (1.0 - x_nh3) * 100

        conversions.append(conversion)
        h2_fractions.append(x_h2 * 100)
        n2_fractions.append(x_n2 * 100)
        nh3_fractions.append(x_nh3 * 100)
    except Exception as e:
        conversions.append(float('nan'))
        h2_fractions.append(float('nan'))
        n2_fractions.append(float('nan'))
        nh3_fractions.append(float('nan'))

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(temperatures_C, conversions, 'b-o', linewidth=2, markersize=4)
ax1.set_xlabel('Temperature (°C)', fontsize=12)
ax1.set_ylabel('NH₃ Conversion (%)', fontsize=12)
ax1.set_title(f'Ammonia Equilibrium Conversion at {pressure} bara')
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 105)
ax1.axhline(y=99.0, color='r', linestyle='--', alpha=0.5, label='99% target')
ax1.legend()

ax2.plot(temperatures_C, h2_fractions, 'g-o', linewidth=2, markersize=4, label='H₂')
ax2.plot(temperatures_C, n2_fractions, 'b-s', linewidth=2, markersize=4, label='N₂')
ax2.plot(temperatures_C, nh3_fractions, 'r-^', linewidth=2, markersize=4, label='NH₃')
ax2.set_xlabel('Temperature (°C)', fontsize=12)
ax2.set_ylabel('Mole Fraction (%)', fontsize=12)
ax2.set_title(f'Equilibrium Composition at {pressure} bara')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('nh3_cracking_temperature.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: nh3_cracking_temperature.png')

## 2. Effect of Pressure on Conversion

According to Le Chatelier's principle, lower pressure favors the decomposition
reaction (2 moles → 4 moles). We study the equilibrium at different pressures.

In [ ]:
# Pressure effect at different temperatures
pressures = [1.0, 5.0, 10.0, 20.0, 50.0]
temperatures_sweep = np.arange(200, 950, 50)

fig, ax = plt.subplots(figsize=(10, 7))

for P in pressures:
    conv_at_P = []

    for t in temperatures_sweep:
        try:
            feed_f = SystemSrkEos(273.15 + float(t), float(P))
            feed_f.addComponent('ammonia', 1.0)
            feed_f.addComponent('nitrogen', 1e-10)
            feed_f.addComponent('hydrogen', 1e-10)
            feed_f.setMixingRule('classic')

            fd = Stream('Feed', feed_f)
            fd.setFlowRate(1000.0, 'kg/hr')
            fd.setTemperature(float(t), 'C')
            fd.setPressure(float(P), 'bara')

            proc = ProcessSystem()
            proc.add(fd)

            rx = GibbsReactor('Rx', fd)
            rx.setEnergyMode('isothermal')
            rx.setMaxIterations(5000)
            proc.add(rx)

            proc.run()

            x_nh3 = float(rx.getOutletStream().getFluid().getPhase(0).getComponent('ammonia').getx())
            conv_at_P.append((1.0 - x_nh3) * 100)
        except Exception:
            conv_at_P.append(float('nan'))

    ax.plot(temperatures_sweep, conv_at_P, 'o-', linewidth=2, markersize=5, label=f'{P:.0f} bara')

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('NH₃ Conversion (%)', fontsize=12)
ax.set_title('Ammonia Decomposition: Effect of Pressure on Equilibrium Conversion')
ax.legend(title='Pressure', fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('nh3_cracking_pressure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: nh3_cracking_pressure.png')

## 3. Complete Ammonia Cracking Process

Simulate a complete cracking process:
1. Liquid NH₃ pump and vaporization
2. Heat exchange between cracker product and feed
3. Gibbs reactor (cracker) at 700°C, 10 bara
4. Product cooling and hydrogen separation

In [ ]:
# Complete NH3 cracking process
# Feed: liquid ammonia at 25°C, 15 bara
nh3_feed_fluid = SystemSrkEos(273.15 + 25.0, 15.0)
nh3_feed_fluid.addComponent('ammonia', 1.0)
nh3_feed_fluid.addComponent('nitrogen', 1e-10)
nh3_feed_fluid.addComponent('hydrogen', 1e-10)
nh3_feed_fluid.setMixingRule('classic')

nh3_feed = Stream('NH3 Liquid Feed', nh3_feed_fluid)
nh3_feed.setFlowRate(10000.0, 'kg/hr')  # 10 t/h NH3
nh3_feed.setTemperature(25.0, 'C')
nh3_feed.setPressure(15.0, 'bara')

process = ProcessSystem()
process.add(nh3_feed)

# Pre-heat NH3 feed to reactor temperature
nh3_heater = Heater('NH3 Vaporizer', nh3_feed)
nh3_heater.setOutTemperature(273.15 + 700.0)
process.add(nh3_heater)

# Gibbs Reactor (cracker) - isothermal at 700°C
cracker = GibbsReactor('NH3 Cracker', nh3_heater.getOutletStream())
cracker.setEnergyMode('isothermal')
cracker.setMaxIterations(5000)
cracker.setConvergenceTolerance(1e-6)
process.add(cracker)

# Cool cracker product
product_cooler = Cooler('Product Cooler', cracker.getOutletStream())
product_cooler.setOutTemperature(273.15 + 40.0)
process.add(product_cooler)

# Run process
process.run()

print('=== NH3 Cracking Process Results ===')
print(f'Feed Rate:               {nh3_feed.getFlowRate("kg/hr"):.0f} kg/hr NH₃')
print(f'Reactor Temperature:     700 °C')
print(f'Reactor Pressure:        {cracker.getOutletStream().getPressure("bara"):.1f} bara')
print()

# Product composition
product = cracker.getOutletStream().getFluid()
print('Product Composition:')
for i in range(int(product.getNumberOfComponents())):
    comp = product.getPhase(0).getComponent(i)
    name = str(comp.getComponentName())
    x = float(comp.getx()) * 100
    if x > 0.001:
        print(f'  {name:15s}: {x:8.3f} mol%')

# Conversion
x_nh3_out = float(product.getPhase(0).getComponent('ammonia').getx())
conversion = (1.0 - x_nh3_out) * 100
print(f'\nNH₃ Conversion:          {conversion:.1f}%')

# H2 production
x_h2_out = float(product.getPhase(0).getComponent('hydrogen').getx())
product_flow = cracker.getOutletStream().getFlowRate('kg/hr')
print(f'Product Flow Rate:       {product_flow:.0f} kg/hr')

# Heating duty
heater_duty = abs(nh3_heater.getDuty()) / 1e6  # MW
print(f'\nVaporizer + Heater Duty: {heater_duty:.2f} MW')
cooler_duty = abs(product_cooler.getDuty()) / 1e6
print(f'Product Cooler Duty:     {cooler_duty:.2f} MW')

## 4. H₂ Production Rate Summary

Calculate hydrogen production rates at different cracking conditions.

In [ ]:
# Summary table at various reactor temperatures
reactor_temps = [400, 500, 600, 700, 800, 900]
reactor_pressure = 10.0

summary_rows = []
for t_rx in reactor_temps:
    try:
        f = SystemSrkEos(273.15 + float(t_rx), reactor_pressure)
        f.addComponent('ammonia', 1.0)
        f.addComponent('nitrogen', 1e-10)
        f.addComponent('hydrogen', 1e-10)
        f.setMixingRule('classic')

        s = Stream('Feed', f)
        s.setFlowRate(10000.0, 'kg/hr')
        s.setTemperature(float(t_rx), 'C')
        s.setPressure(reactor_pressure, 'bara')

        p = ProcessSystem()
        p.add(s)

        rx = GibbsReactor('Rx', s)
        rx.setEnergyMode('isothermal')
        rx.setMaxIterations(5000)
        p.add(rx)

        p.run()

        out_fluid = rx.getOutletStream().getFluid()
        x_nh3 = float(out_fluid.getPhase(0).getComponent('ammonia').getx())
        x_h2 = float(out_fluid.getPhase(0).getComponent('hydrogen').getx())
        x_n2 = float(out_fluid.getPhase(0).getComponent('nitrogen').getx())
        conv = (1.0 - x_nh3) * 100

        # H2 mass fraction estimate
        # At full conversion: 17g NH3 -> 1.5g H2 + 14g N2
        # H2 mass fraction = 3*2/(2*17) = 17.6% at full conversion
        h2_mass_frac = x_h2 * 2.016 / (x_h2 * 2.016 + x_n2 * 28.014 + x_nh3 * 17.031)
        h2_kg_hr = 10000.0 * h2_mass_frac

        summary_rows.append({
            'T_reactor (°C)': t_rx,
            'P (bara)': reactor_pressure,
            'Conversion (%)': round(conv, 1),
            'H₂ mol%': round(x_h2 * 100, 1),
            'N₂ mol%': round(x_n2 * 100, 1),
            'NH₃ slip mol%': round(x_nh3 * 100, 2),
            'H₂ production (kg/hr)': round(h2_kg_hr, 0)
        })
    except Exception as e:
        print(f'Warning at {t_rx}°C: {e}')

df_summary = pd.DataFrame(summary_rows)
print('=== NH₃ Cracking - H₂ Production Summary ===')
print(f'NH₃ Feed: 10,000 kg/hr, Pressure: {reactor_pressure} bara')
print()
print(df_summary.to_string(index=False))

## Summary

Key findings from ammonia cracking simulation:

- **Temperature > 600°C** is needed for >95% conversion at 10 bara
- **Lower pressure** shifts equilibrium towards decomposition (Le Chatelier)
- **At 700°C, 10 bara**: near-complete conversion, product is ~75% H₂ + 25% N₂
- **Maximum H₂ yield**: 17.6% of NH₃ mass → H₂ (theoretical limit)
- The endothermic reaction requires significant heat input (~2.7 MJ/kg NH₃)

### Industrial Context
- NH₃ cracking is being developed for green hydrogen delivery to end-users
- Challenges: high temperature requirement, catalyst durability, H₂ purification
- PSA (Pressure Swing Adsorption) needed to separate H₂ from N₂
- Overall energy efficiency: ~60-70% (NH₃ LHV → H₂ LHV)

### Next Steps
- Add PSA model for H₂/N₂ separation
- Heat integration with ammonia vaporization
- Compare with partial oxidation route
- Economic analysis (levelized cost of hydrogen)